# Affiner NLLB-200 pour la traduction français → mooré

Dans ce tutoriel, vous affinez (*fine-tuning*) le modèle de traduction
[NLLB-200 distillé 600M](https://huggingface.co/facebook/nllb-200-distilled-600M) de Meta sur un corpus
parallèle français–mooré, puis vous publiez le modèle obtenu sur le Hugging Face Hub.

## Vue d'ensemble

NLLB-200 (*No Language Left Behind*) est un modèle de traduction multilingue qui couvre 200 langues, dont le
mooré (code `mos_Latn`). Il traduit déjà le français vers le mooré, mais avec une qualité limitée, car le
mooré est peu présent dans ses données d'entraînement. Vous allez l'affiner sur des paires de phrases
français–mooré pour améliorer ses traductions.

Vous allez :

1. charger le corpus parallèle [`burkimbia/moore-web-parallel`](https://huggingface.co/datasets/burkimbia/moore-web-parallel)
   (33 835 paires d'entraînement, 1 992 de validation et 2 074 de test) ;
2. tester le modèle de base sur quelques phrases pour avoir un point de comparaison ;
3. tokeniser le corpus ;
4. entraîner le modèle pendant une époque avec le `Seq2SeqTrainer` de 🤗 Transformers, en mesurant les scores
   [BLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu) et [chrF](https://huggingface.co/spaces/evaluate-metric/chrf),
   et en suivant les courbes avec [Trackio](https://huggingface.co/docs/trackio) ;
5. évaluer le modèle affiné sur le jeu de test, comparer ses traductions à celles du modèle de base, puis le publier sur le Hub.

## Prérequis

**Logiciels**

- Le dépôt `first-experiments` cloné, avec l'environnement installé par `uv sync`, et ce notebook ouvert avec le noyau `.venv` du dépôt.
  Sur Google Colab, la première cellule installe exactement les versions figées dans le fichier `uv.lock` du
  dépôt (environ 5 minutes), puis redémarre la session : relancez alors cette même cellule et continuez.
- Un compte [Hugging Face](https://huggingface.co/join) et un [token d'accès](https://huggingface.co/settings/tokens) avec les droits **write** :
  - en local, placez-le dans un fichier `.env` à la racine du dépôt : `HF_TOKEN=hf_...` ;
  - sur Colab, ajoutez-le dans les secrets (icône 🔑) sous le nom `HF_TOKEN`.
- Un accès au jeu de données [`burkimbia/moore-web-parallel`](https://huggingface.co/datasets/burkimbia/moore-web-parallel). Il est publié sous
  licence CC BY-NC 4.0 (usage non commercial) et l'accès est validé manuellement : ouvrez sa page, remplissez
  le formulaire de demande, puis attendez l'approbation avant de lancer ce notebook.

**Tutoriels préalables**

Aucun. Une connaissance de base de Python et de PyTorch suffit.

## Ressources nécessaires

| Ressource | Besoin |
| --- | --- |
| GPU | **Obligatoire** : GPU NVIDIA avec au moins **16 Go** de mémoire (par exemple un T4 sur Colab). Un GPU de portable de 4 à 8 Go ne suffit pas pour l'entraînement. |
| Mémoire vive | 12 Go |
| Disque | Environ 8 Go (modèle de 2,5 Go et point de sauvegarde d'environ 2,5 Go) |
| Réseau | Téléchargement d'environ 2,5 Go (modèle et corpus) |

## Durée

- **Lecture et exécution des cellules** : environ 20 minutes.
- **Calcul** : environ 1 h 30 à 2 h d'entraînement sur un T4, et environ 30 minutes sur un A100.

## 1. Préparer l'environnement

Cette cellule charge votre token Hugging Face, en local comme sur Colab.

In [ ]:
import os
import subprocess

if "COLAB_RELEASE_TAG" in os.environ:
    # Google Colab : installez exactement les versions figées dans uv.lock, puis redémarrez la session.
    if not os.path.exists("/content/.env_ready"):
        repo_dir = "/content/first-experiments"
        subprocess.run(["pip", "install", "-q", "uv"], check=True)
        if not os.path.exists(repo_dir):
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1", "https://github.com/BurkimbIA/first-experiments.git", repo_dir],
                check=True,
            )
        subprocess.run(
            ["uv", "export", "--frozen", "--no-dev", "--no-hashes", "--no-emit-project", "-o", "/content/requirements.txt"],
            cwd=repo_dir,
            check=True,
        )
        subprocess.run(
            [
                "uv", "pip", "install", "--system", "-q", "-r", "/content/requirements.txt",
                "--extra-index-url", "https://download.pytorch.org/whl/cu126",
                "--index-strategy", "unsafe-best-match",
            ],
            check=True,
        )
        open("/content/.env_ready", "w").close()
        print("Installation terminée. La session redémarre : relancez cette cellule.")
        os.kill(os.getpid(), 9)

    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Machine locale : le token est lu depuis le fichier .env à la racine du dépôt.
    from dotenv import load_dotenv

    load_dotenv()

Vérifiez que vous êtes bien connecté au Hub et qu'un GPU est disponible. Le nom de compte affiché sert plus
loin à nommer le dépôt du modèle.

In [ ]:
import torch
from huggingface_hub import whoami

hf_user = whoami()["name"]
print("Compte Hugging Face :", hf_user)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun GPU détecté")

## 2. Charger le corpus parallèle

Le corpus contient une colonne `french`, une colonne `moore`, la provenance de chaque paire (`source`) et des
scores de qualité calculés automatiquement (`laser_score`, `comet_qe`, `len_ratio`).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("burkimbia/moore-web-parallel")
dataset

In [ ]:
dataset["train"].to_pandas().sample(5, random_state=0)[["french", "moore", "source"]]

## 3. Charger le modèle et le tokeniseur

NLLB identifie chaque langue par un code : `fra_Latn` pour le français et `mos_Latn` pour le mooré. Vous les
passez au tokeniseur, qui ajoute alors le bon code de langue au début de chaque phrase.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"
src_lang = "fra_Latn"
tgt_lang = "mos_Latn"

tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang=src_lang, tgt_lang=tgt_lang)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Regardez comment le tokeniseur encode une paire de phrases : la source commence par `fra_Latn`, la cible
(`labels`) par `mos_Latn`, et chacune se termine par le jeton de fin `</s>`.

In [ ]:
example = tokenizer("Tu es mon semblable.", text_target="Foo la mam bogre.")

print(tokenizer.decode(example["input_ids"]))
print(tokenizer.decode(example["labels"]))

Pendant la génération, vous devez forcer le premier jeton produit à `mos_Latn`. Sinon, le modèle peut
répondre dans une autre langue. Enregistrez ce réglage dans la configuration de génération du modèle : il
sera utilisé pendant l'évaluation, puis publié avec le modèle.

In [ ]:
model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
model.generation_config.max_length = 256

## 4. Tester le modèle de base

Avant l'entraînement, traduisez quelques phrases du jeu de test. Vous comparerez ces traductions à celles du
modèle affiné à la fin du tutoriel.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


def translate(texts, model):
    inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


samples = dataset["test"][:3]
baseline_translations = translate(samples["french"], model)

for french, reference, prediction in zip(samples["french"], samples["moore"], baseline_translations):
    print(f"Français   : {french}")
    print(f"Référence  : {reference}")
    print(f"Modèle base: {prediction}")
    print()

## 5. Tokeniser le corpus

Tokenisez toutes les paires. Les phrases trop longues sont tronquées à 256 jetons. Vous n'ajoutez pas de
remplissage (*padding*) à cette étape : le `DataCollatorForSeq2Seq` complète chaque lot jusqu'à la phrase
la plus longue du lot, ce qui est beaucoup plus rapide qu'un remplissage fixe à 256 jetons.

In [ ]:
from transformers import DataCollatorForSeq2Seq


def tokenize_fn(batch):
    return tokenizer(batch["french"], text_target=batch["moore"], max_length=256, truncation=True)


tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenisation",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
tokenized_dataset

## 6. Définir les métriques

Vous mesurez deux scores entre 0 et 100, où un score plus élevé est meilleur :

- **BLEU** compte les suites de mots communes entre la traduction et la référence ;
- **chrF** compte les suites de caractères communes. Il est plus adapté au mooré, dont l'orthographe et les tons
  font varier les mots.

Les positions de remplissage, marquées `-100`, sont remplacées par le jeton de remplissage avant le décodage.

In [ ]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = [[label] for label in tokenizer.batch_decode(labels, skip_special_tokens=True)]

    return {
        "bleu": bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"],
        "chrf": chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"],
    }

## 7. Configurer l'entraînement

Les paramètres principaux sont les suivants :

- **`train_sampling_strategy="group_by_length"`** : forme des lots de phrases de longueurs proches. Le
  remplissage est ainsi réduit, et l'entraînement va plus vite. Les longueurs sont calculées une fois au
  début de l'entraînement, à partir des `input_ids` ;
- **taille de lot effective de 8** : 4 exemples par lot × 2 pas d'accumulation de gradient ;
- **optimiseur Adafactor** : contrairement à AdamW, il ne garde pas de moyenne des gradients (`beta1=None`) et
  stocke une version compressée de leur variance. Il utilise donc beaucoup moins de mémoire GPU et produit des
  points de sauvegarde plus légers. Il est souvent utilisé pour affiner les modèles de traduction comme NLLB.
  Avec `relative_step=False`, il suit le taux d'apprentissage fixe de `2e-5` ;
- **`activation_checkpointing`** : réduit la mémoire utilisée en échange d'un calcul un peu plus long ;
- **`bf16` ou `fp16`** : précision mixte. Le `bf16` est utilisé si le GPU le prend en charge (A100, L4), sinon le
  `fp16` (T4) ;
- **évaluation tous les 1 000 pas** : chaque évaluation traduit tout le jeu de validation et prend plusieurs minutes ;
- **`report_to="trackio"`** : envoie les courbes de perte et de métriques à Trackio ;
- **`push_to_hub=True`, `hub_strategy="end"`** : le modèle est publié dans un dépôt privé à la fin de l'entraînement.

Si vous obtenez une erreur de mémoire (`CUDA out of memory`), passez `per_device_train_batch_size` à 2 et
`gradient_accumulation_steps` à 4 : la taille de lot effective reste de 8.

In [ ]:
from transformers import Seq2SeqTrainingArguments

repo_name = "nllb-200-distilled-600M-fra-mos"
hub_model_id = f"{hf_user}/{repo_name}"
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = Seq2SeqTrainingArguments(
    output_dir=f"outputs/{repo_name}",
    train_sampling_strategy="group_by_length",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    activation_checkpointing=True,
    learning_rate=2e-5,
    lr_scheduler_type="constant",
    bf16=use_bf16,
    fp16=not use_bf16,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=256,
    report_to="trackio",
    project="burkimbia-nllb-fra-mos",
    run_name="nllb-600M-fra-mos-1-epoch",
    push_to_hub=True,
    hub_model_id=hub_model_id,
    hub_private_repo=True,
    hub_strategy="end",
)

## 8. Lancer l'entraînement

Créez l'optimiseur Adafactor, puis le `Seq2SeqTrainer`. Le trainer crée lui-même le planificateur du taux
d'apprentissage (`None`), ici constant. Passez le tokeniseur à `processing_class` : il sera enregistré et publié avec le
modèle, ce qui rend le modèle publié directement utilisable.

In [ ]:
from transformers import Seq2SeqTrainer
from transformers.optimization import Adafactor

optimizer = Adafactor(
    model.parameters(),
    lr=training_args.learning_rate,
    eps=(1e-30, 1e-3),
    clip_threshold=1.0,
    decay_rate=-0.8,
    beta1=None,
    scale_parameter=False,
    relative_step=False,
    warmup_init=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)

Lancez l'entraînement. Il dure environ 1 h 30 à 2 h sur un T4.

Pour suivre les courbes en direct sur une machine locale, ouvrez un terminal à la racine du dépôt et lancez :

```bash
uv run trackio show --project burkimbia-nllb-fra-mos
```

In [ ]:
trainer.train()

## 9. Évaluer le modèle affiné

Calculez les scores BLEU et chrF sur le jeu de test, que le modèle n'a jamais vu pendant l'entraînement.

In [ ]:
test_metrics = trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")
print(f"BLEU (test) : {test_metrics['test_bleu']:.2f}")
print(f"chrF (test) : {test_metrics['test_chrf']:.2f}")

Traduisez à nouveau les trois phrases de l'étape 4 et comparez les résultats avec ceux du modèle de base.

In [ ]:
finetuned_translations = translate(samples["french"], trainer.model)

for french, reference, before, after in zip(
    samples["french"], samples["moore"], baseline_translations, finetuned_translations
):
    print(f"Français     : {french}")
    print(f"Référence    : {reference}")
    print(f"Modèle base  : {before}")
    print(f"Modèle affiné: {after}")
    print()

## 10. Publier le modèle sur le Hub

Publiez le modèle, le tokeniseur et une fiche de modèle générée automatiquement dans le dépôt privé
`<votre compte>/nllb-200-distilled-600M-fra-mos`. Trackio publie aussi les courbes d'entraînement dans un
Space Hugging Face, accessible depuis l'onglet « Training Metrics » de la page du modèle.

In [ ]:
trainer.push_to_hub()
print(f"Modèle publié : https://huggingface.co/{hub_model_id}")

Pour vérifier la publication, rechargez le modèle depuis le Hub et traduisez une nouvelle phrase. La
configuration de génération publiée force déjà la sortie en mooré.

In [ ]:
hub_model = AutoModelForSeq2SeqLM.from_pretrained(hub_model_id).to(device)
print(translate(["Le marché ouvre demain matin."], hub_model)[0])